# ★ 최종 제출 — 2026-08-31

## 반드시 Save & Run All (커밋)로 실행하세요

대화형 세션은 유휴 시 "Are you still here?"가 뜨고 **Continue를 누르면 커널이 재시작**됩니다.
5시간 40분을 화면 앞에서 지킬 수 없으므로 **커밋이 유일한 방법**입니다.

## 사전 준비
1. 구글 드라이브에서 `test_submission.csv` 다운로드
2. **Create → New Dataset**으로 업로드 (이름 예: `deep-chal-test`)
3. 이 노트북 **+ Add Input**에 그 Dataset만 추가 (대회 데이터·어댑터는 **불필요**)
4. Settings: **Accelerator = GPU T4 x2**, **Internet = On**
5. **Save Version → Save & Run All (Commit)**
6. 컴퓨터 꺼도 됩니다. 약 **5시간 40분** 뒤 완료

## 설정 근거

| 항목 | 값 | 근거 |
|---|---|---|
| 모델 | Qwen2.5-3B-Instruct (베이스) | 앙상블은 리더보드 0.77737로 하락 |
| 샘플 수 | **32** | N=64는 0.78339로 이득 없음. N=32가 **0.78580**으로 최고 |
| temperature | 0.8 | 0.7과 결과 동일 |
| 어댑터 | **미사용** | v2 어댑터는 로컬·리더보드 모두 하락 |

## 제출 형식 (공지 준수)
> 다운로드받으신 테스트 데이터셋의 **answer 컬럼에** 추론한 정수 답을 입력하여 업로드
> answer 컬럼에는 **'정수'만** 포함되어야 합니다

→ 원본 `id, question, answer` 구조를 유지하고 `answer`만 채웁니다.

## 규칙 확인
| 규칙 | 상태 |
|---|---|
| 4.1a 베이스 모델 고정 | O Qwen2.5-3B-Instruct |
| 4.3 외부 모델 앙상블 금지 | O 없음 |
| 추론 시 인터넷·코드 실행 금지 | O 모델 출력만 사용 |
| 5.1b test를 학습에 사용 금지 | O 추론만 |
| **5.2c 외부 데이터셋 명시** | ▲ **[8] 셀 출력을 구글 폼에 기재** |


---
## [1] 설정

In [1]:
N_SAMPLES  = 32       # 검증된 최적값 (리더보드 0.78580)
TEMP       = 0.8
MAX_TOKENS = 1024
CHUNK      = 100      # 2000문제 -> 20덩어리, 덩어리당 약 17분
SEED       = 42
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
WORK       = "/kaggle/working"
SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")
print(f"N_SAMPLES={N_SAMPLES} | CHUNK={CHUNK}")

N_SAMPLES=32 | CHUNK=100


---
## [2] 설치 — 커밋 모드용, 재시작 불필요

Kaggle 커널은 부팅 시 **protobuf 5.29.5를 이미 메모리에 올려둡니다.**
그래서 디스크를 6.x로 올리면 충돌합니다. → **디스크를 메모리에 맞춥니다.**

`ray`와 `opentelemetry`가 protobuf 6.x용 코드를 들고 오는 범인이라 제거합니다.
둘 다 GPU 한 장 추론에는 필요 없고, 없으면 vLLM이 알아서 건너뜁니다.

In [2]:
!pip install -q -U vllm 2>&1 | tail -1
!pip uninstall -q -y ray opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp
!pip install -q --force-reinstall "protobuf==5.29.5" 2>&1 | tail -1

import google.protobuf as p, torch
print("protobuf:", p.__version__)
print("GPU    :", torch.cuda.device_count())
assert torch.cuda.device_count() > 0, "GPU 없음 - Settings에서 Accelerator를 켜세요"
from vllm import LLM
print("vllm import 성공")

gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.6.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.44.0 which is incompatible.
protobuf: 5.29.5
GPU    : 2
vllm import 성공


---
## [3] test 데이터 로드

`/kaggle/input/` 아래에서 test csv를 찾고, **원본 컬럼 구조를 보존**합니다.

In [3]:
import glob, os
import pandas as pd

cands = [q for q in glob.glob("/kaggle/input/**/*.csv", recursive=True)
         if "test" in os.path.basename(q).lower()]
print("후보:", cands)
assert len(cands) == 1, f"test csv가 {len(cands)}개 발견됨. Dataset을 정확히 하나만 붙이세요"
TEST_PATH = cands[0]

work = pd.read_csv(TEST_PATH)
ORIG_COLS = list(work.columns)
print("\n파일:", TEST_PATH)
print("shape:", work.shape, "| 컬럼:", ORIG_COLS)
assert "id" in work.columns and "question" in work.columns
assert work["id"].is_unique, "id 중복이 있습니다"
print(f"\n문제 {len(work):,}개 | id: {work['id'].iloc[0]} ~ {work['id'].iloc[-1]}")

n_chunk = (len(work) + CHUNK - 1) // CHUNK
print(f"{n_chunk}개 덩어리 | 예상 소요 약 {len(work)*N_SAMPLES*0.316/3600:.1f}시간")
work.head(2)

후보: ['/kaggle/input/datasets/mxinjxe/deep-chal-test/test_submission.csv']

파일: /kaggle/input/datasets/mxinjxe/deep-chal-test/test_submission.csv
shape: (2000, 3) | 컬럼: ['id', 'question', 'answer']

문제 2,000개 | id: test-0000 ~ test-1999
20개 덩어리 | 예상 소요 약 5.6시간


,id,question,answer
0,test-0000,The average mark of the students of a class in...,NaN
1,test-0001,How many digits are in the base $10$ represent...,NaN


---
## [4] 답 추출기

리더보드 0.78580을 낸 것과 동일. 자체 검증 실패 시 `assert`로 멈춥니다.

In [4]:
import re
from collections import Counter

def extract_boxed(text):
    """마지막 \\boxed{...} 안의 원문을 중괄호 짝을 맞춰 추출"""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # \boxed 15 형태
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/텍스트 -> 파이썬 int. float()을 쓰지 않아 초대형 정수도 보존."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None = 이 샘플은 유효한 정수를 못 냈음 -> 투표에서 기권"""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]

_c = [(r"\boxed{132}",132), (r"\boxed{-2,025,078}",-2025078),
      (r"\boxed{3{,}431{,}577{,}212{,}128{,}939}",3431577212128939),
      (r"\boxed{\dfrac{650}{5}}",130), (r"\boxed{42 \text{ cm}}",42),
      (r"\boxed{\left(\frac{100}{4}\right)}",25),
      (r"\boxed{\frac{7}{2}}",None), (r"no box, ends with 12 cats",12)]
bad = sum(parse_answer(t) != w for t, w in _c)
print("parser FAILURES:", bad, "/", len(_c))
assert bad == 0, "파서 검증 실패 - 진행하지 마세요"

parser FAILURES: 0 / 8


---
## [5] 모델 로드 — 5~10분

어댑터를 쓰지 않으므로 `enable_lora`도 켜지 않습니다. 순수 베이스 모델입니다.

In [5]:
import time, json
import numpy as np
from collections import Counter
from vllm import SamplingParams
from transformers import AutoTokenizer

llm = LLM(model=MODEL_ID, dtype="half", max_model_len=4096,
          gpu_memory_utilization=0.90, tensor_parallel_size=1,
          seed=SEED, trust_remote_code=True)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
sp  = SamplingParams(n=N_SAMPLES, temperature=TEMP, top_p=0.95,
                     max_tokens=MAX_TOKENS, seed=SEED)

def make_prompts(qs):
    return [tok.apply_chat_template(
        [{"role":"system","content":SYSTEM},{"role":"user","content":q}],
        tokenize=False, add_generation_prompt=True) for q in qs]

print("로드 완료")

INFO 08-31 01:03:09 [api_utils.py:272] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'seed': 42, 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

INFO 08-31 01:03:26 [model.py:672] Resolved architecture: Qwen2ForCausalLM
WARNING 08-31 01:03:26 [model.py:2299] Casting torch.bfloat16 to torch.float16.
INFO 08-31 01:03:26 [model.py:1965] Using max model len 4096
INFO 08-31 01:03:26 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-31 01:03:26 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

(EngineCore pid=197) INFO 08-31 01:04:10 [default_loader.py:430] Loading weights took 6.20 seconds
(EngineCore pid=197) INFO 08-31 01:04:11 [model_runner.py:380] Model loading took 5.79 GiB memory and 38.778703 seconds
(EngineCore pid=197) WARNING 08-31 01:04:11 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=197) INFO 08-31 01:04:25 [backends.py:1094] Using cache directory: /root/.cache/vllm/torch_compile_cache/2641bb3ea4/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=197) INFO 08-31 01:04:25 [backends.py:1155] Dynamo bytecode transform time: 10.23 s


(EngineCore pid=197) [rank0]:W0831 01:04:31.289000 197 torch/_inductor/utils.py:1953] Not enough SMs to use max_autotune_gemm mode


(EngineCore pid=197) INFO 08-31 01:04:34 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 9.05 s
(EngineCore pid=197) INFO 08-31 01:04:39 [backends.py:920] collected artifacts: 37 entries, 3 artifacts, 4644799 bytes total
(EngineCore pid=197) INFO 08-31 01:04:39 [decorators.py:708] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/e50f6a777fcfc0decb70521dcde71ec9fcdecc38332dd56e3d25121b7d4171eb/rank_0_0/model
(EngineCore pid=197) INFO 08-31 01:04:39 [monitor.py:53] torch.compile took 24.71 s in total
(EngineCore pid=197) INFO 08-31 01:04:39 [monitor.py:81] Initial profiling/warmup run took 0.11 s
(EngineCore pid=197) INFO 08-31 01:04:43 [gpu_worker.py:578] Available KV cache memory: 5.81 GiB
(EngineCore pid=197) INFO 08-31 01:04:43 [kv_cache_utils.py:1869] GPU KV cache size: 169,280 tokens, Maximum concurrency for 4,096 tokens per request: 41.33x


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:02<00:00, 13.42it/s]


(EngineCore pid=197) INFO 08-31 01:04:56 [model_runner.py:906] Graph capturing finished in 13 secs, took 0.54 GiB
(EngineCore pid=197) INFO 08-31 01:04:56 [gpu_worker.py:804] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 6.69 GiB for consumed memory (weights + non-torch), 0.6 GiB for peak activation, and 0.54 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=5504613581` (5.13 GiB) to fit into requested memory, or `--kv-cache-memory=6958247936` (6.48 GiB) to fully utilize gpu memory. Current kv cache memory in use is 5.81 GiB.
(EngineCore pid=197) INFO 08-31 01:05:05 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=197) INFO 08-31 01:05:06 [torch_utils.py:262] Reducing Torch threads from 2 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=197) INFO 08-3

---
## ★ [6] 생성 — 약 5시간 40분

100문제씩 20덩어리로 나눠 각 결과를 저장합니다.
**이미 저장된 덩어리는 건너뛰므로**, 대화형에서 끊겼을 때 이 셀만 다시 실행하면 이어집니다.

In [6]:
os.makedirs(f"{WORK}/chunks", exist_ok=True)
cpath = lambda i: f"{WORK}/chunks/chunk_{i:03d}.json"

t_start = time.time()
for ci in range(n_chunk):
    if os.path.exists(cpath(ci)):
        print(f"[{ci+1}/{n_chunk}] 이미 저장됨 - 건너뜀"); continue

    sub_df  = work.iloc[ci*CHUNK : (ci+1)*CHUNK]
    prompts = make_prompts(sub_df["question"])
    t0 = time.time()

    outs = llm.generate(prompts, sp, use_tqdm=False)
    vals = [[parse_answer(c.text) for c in o.outputs] for o in outs]

    with open(cpath(ci), "w") as f:
        json.dump({"ids": list(sub_df["id"]), "vals": vals}, f)

    el, done = time.time() - t0, ci + 1
    print(f"[{done}/{n_chunk}] {len(sub_df)}문제 {el/60:.1f}분 | "
          f"누적 {(time.time()-t_start)/60:.0f}분 | 남은 예상 {(n_chunk-done)*el/60:.0f}분")

print(f"\n생성 완료: {(time.time()-t_start)/60:.1f}분")

[1/20] 100문제 18.8분 | 누적 19분 | 남은 예상 358분
[2/20] 100문제 17.6분 | 누적 36분 | 남은 예상 316분
[3/20] 100문제 19.5분 | 누적 56분 | 남은 예상 332분
[4/20] 100문제 20.2분 | 누적 76분 | 남은 예상 323분
[5/20] 100문제 19.8분 | 누적 96분 | 남은 예상 297분
[6/20] 100문제 20.4분 | 누적 116분 | 남은 예상 285분
[7/20] 100문제 17.0분 | 누적 133분 | 남은 예상 221분
[8/20] 100문제 18.9분 | 누적 152분 | 남은 예상 227분
[9/20] 100문제 18.3분 | 누적 170분 | 남은 예상 201분
[10/20] 100문제 18.5분 | 누적 189분 | 남은 예상 185분
[11/20] 100문제 18.1분 | 누적 207분 | 남은 예상 163분
[12/20] 100문제 20.7분 | 누적 228분 | 남은 예상 166분
[13/20] 100문제 21.3분 | 누적 249분 | 남은 예상 149분
[14/20] 100문제 19.6분 | 누적 269분 | 남은 예상 118분
[15/20] 100문제 18.0분 | 누적 287분 | 남은 예상 90분
[16/20] 100문제 20.2분 | 누적 307분 | 남은 예상 81분
[17/20] 100문제 18.4분 | 누적 325분 | 남은 예상 55분
[18/20] 100문제 17.1분 | 누적 342분 | 남은 예상 34분
[19/20] 100문제 20.1분 | 누적 363분 | 남은 예상 20분
[20/20] 100문제 20.3분 | 누적 383분 | 남은 예상 0분

생성 완료: 382.8분


---
## [7] 다수결 + 원본 형식으로 저장

**원본 파일의 `answer` 컬럼만 채웁니다.** 컬럼 구성과 순서는 그대로 둡니다.
`answer`는 반드시 **int64**여야 합니다. float이면 `132.0`처럼 저장되어 오답 처리됩니다.

In [7]:
def majority(vals, fb=0):
    v = [x for x in vals if x is not None]
    return Counter(v).most_common(1)[0][0] if v else fb

pred_map, n_fail, n_tot, shares = {}, 0, 0, []
for ci in range(n_chunk):
    assert os.path.exists(cpath(ci)), f"덩어리 {ci} 없음 - [6]을 다시 실행하세요"
    d = json.load(open(cpath(ci)))
    for _id, vals in zip(d["ids"], d["vals"]):
        pred_map[_id] = majority(vals)
        n_fail += sum(v is None for v in vals); n_tot += len(vals)
        cnt = Counter([v for v in vals if v is not None])
        shares.append(cnt.most_common(1)[0][1]/len(vals) if cnt else 0)

print(f"문제 {len(pred_map):,} | 샘플 {n_tot:,}")
print(f"파싱 실패율 {n_fail/n_tot:.2%} | 평균 득표율 {np.mean(shares):.3f}")
print("(리더보드 기준: 파싱실패 1.70%, 득표율 0.726)")

out = work.copy()
out["answer"] = [int(pred_map[i]) for i in out["id"]]
out["answer"] = out["answer"].astype("int64")
out = out[ORIG_COLS]
out.to_csv(f"{WORK}/submission.csv", index=False)

print(f"\nsubmission.csv 저장: {out.shape}")
print(open(f"{WORK}/submission.csv").read()[:300])

문제 2,000 | 샘플 64,000
파싱 실패율 0.86% | 평균 득표율 0.671
(리더보드 기준: 파싱실패 1.70%, 득표율 0.726)

submission.csv 저장: (2000, 3)
id,question,answer
test-0000,"The average mark of the students of a class in a particular exam is some value. If 5 students whose average mark in that exam is 44 are excluded, the average mark of the remaining will be 80. There were 9 students who wrote the exam. What was the initial average mark of


---
## [8] 최종 점검 + 제출 문구

**하나라도 FAIL이면 제출하지 마세요.**

In [8]:
ok = True
def check(c, m):
    global ok
    print(("  OK   " if c else "  FAIL ") + m); ok = ok and c

print("제출 파일 점검")
check(list(out.columns) == ORIG_COLS, f"컬럼이 원본과 동일 {ORIG_COLS}")
check(len(out) == len(work), f"행 수 {len(out)} == 원본 {len(work)}")
check(out["id"].is_unique, "id 중복 없음")
check(list(out["id"]) == list(work["id"]), "id 순서가 원본과 동일")
check(out["answer"].notna().all(), "answer 빈 값 없음")
check(str(out["answer"].dtype) == "int64", f"answer 자료형 int64 (현재 {out['answer'].dtype})")
check(out["answer"].map(lambda v: float(v).is_integer()).all(), "answer 전부 정수")
check(n_fail/n_tot < 0.10, f"파싱 실패율 {n_fail/n_tot:.2%} < 10%")

print("\n" + ("★ 전체 통과 - 제출 가능" if ok else "★ FAIL 항목 있음 - 제출 금지"))

print("\n" + "="*72)
print("구글 폼에 기재할 내용 (규칙 5.2c)")
print("="*72)
print(r"""
[사용한 외부 데이터셋]
- AI-MO/NuminaMath-1.5 (Apache 2.0)
  https://huggingface.co/datasets/AI-MO/NuminaMath-1.5
  용도: SFT 학습 데이터 (실험 단계에서만 사용, 최종 제출 모델에는 미적용)
  처리: math-word-problem 중 정수 답 문항만 필터링, 무작위 추출(source별 비율 유지),
        평가 문항(로컬 검증 300 + 리더보드 831)은 사전 제거

[최종 제출 모델 구성]
- 베이스: Qwen/Qwen2.5-3B-Instruct (가중치 변경 없음, 파인튜닝 미적용)
- 추론: 문제당 32샘플 생성 후 다수결 (Self-Consistency)
        temperature 0.8, top_p 0.95, max_tokens 1024
- 후처리: 생성 텍스트에서 마지막 \boxed{} 내 정수를 추출,
          정수가 아니면 해당 샘플을 투표에서 제외
- 코드 실행 / 도구 호출 / 인터넷 접속 없음
""")

from IPython.display import FileLink
FileLink("submission.csv")

제출 파일 점검
  OK   컬럼이 원본과 동일 ['id', 'question', 'answer']
  OK   행 수 2000 == 원본 2000
  OK   id 중복 없음
  OK   id 순서가 원본과 동일
  OK   answer 빈 값 없음
  OK   answer 자료형 int64 (현재 int64)
  OK   answer 전부 정수
  OK   파싱 실패율 0.86% < 10%

★ 전체 통과 - 제출 가능

구글 폼에 기재할 내용 (규칙 5.2c)

[사용한 외부 데이터셋]
- AI-MO/NuminaMath-1.5 (Apache 2.0)
  https://huggingface.co/datasets/AI-MO/NuminaMath-1.5
  용도: SFT 학습 데이터 (실험 단계에서만 사용, 최종 제출 모델에는 미적용)
  처리: math-word-problem 중 정수 답 문항만 필터링, 무작위 추출(source별 비율 유지),
        평가 문항(로컬 검증 300 + 리더보드 831)은 사전 제거

[최종 제출 모델 구성]
- 베이스: Qwen/Qwen2.5-3B-Instruct (가중치 변경 없음, 파인튜닝 미적용)
- 추론: 문제당 32샘플 생성 후 다수결 (Self-Consistency)
        temperature 0.8, top_p 0.95, max_tokens 1024
- 후처리: 생성 텍스트에서 마지막 \boxed{} 내 정수를 추출,
          정수가 아니면 해당 샘플을 투표에서 제외
- 코드 실행 / 도구 호출 / 인터넷 접속 없음



/kaggle/working/submission.csv